In [1]:
!pip install -q gradio

In [2]:
import os

print(os.path.exists('/content/drive/MyDrive/SI26-urdu-ocr-model'))
print(os.listdir('/content/drive/MyDrive'))

True
['Colab Notebooks', 'CamScanner 12-01-2023 16.04.pdf', 'CamScanner 12-01-2023 16.05.pdf', 'CamScanner 12-01-2023 16.06.pdf', 'CamScanner 12-01-2023 16.06 (1).pdf', 'CamScanner 12-01-2023 16.07.pdf', 'CamScanner 12-01-2023 16.08.pdf', 'CamScanner 12-01-2023 16.09.pdf', 'CamScanner 12-01-2023 16.09 (1).pdf', 'CamScanner 12-01-2023 16.09 (2).pdf', 'CamScanner 12-01-2023 16.10.pdf', 'CamScanner 12-01-2023 16.10 (1).pdf', 'CamScanner 12-01-2023 16.10 (2).pdf', 'CamScanner 12-01-2023 16.11.pdf', 'CamScanner 12-01-2023 16.11 (1).jpg', 'CamScanner 12-01-2023 16.10 (2) (1).jpg', 'CamScanner 12-01-2023 16.10 (1) (1).jpg', 'CamScanner 12-01-2023 16.10 (3).jpg', 'CamScanner 12-01-2023 16.09 (2) (1).jpg', 'CamScanner 12-01-2023 16.09 (1) (1).jpg', 'CamScanner 12-01-2023 16.09.jpg', 'CamScanner 12-01-2023 16.08.jpg', 'CamScanner 12-01-2023 16.07.jpg', 'CamScanner 12-01-2023 16.06 (1).jpg', 'CamScanner 12-01-2023 16.06.jpg', 'CamScanner 12-01-2023 16.05.jpg', 'CamScanner 12-01-2023 16.04.jpg', '

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Tesla T4


In [4]:
import gradio as gr
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import torch

model_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"

# Load processor from local folder
processor = TrOCRProcessor.from_pretrained(
    model_path,
    local_files_only=True
)

# Load model from local folder
model = VisionEncoderDecoderModel.from_pretrained(
    model_path,
    local_files_only=True
)

model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded successfully!")
print("Device:", device)

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda


In [5]:
def extract_urdu_text(image):
    if image is None:
        return "Please upload an image."

    # Convert image to RGB
    image = image.convert("RGB")

    # Process image
    pixel_values = processor(
        images=image,
        return_tensors="pt"
    ).pixel_values

    pixel_values = pixel_values.to(device)

    # Generate prediction
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=128
        )

    # Decode text
    text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    if text.strip():
        return text

    return "Could not extract text from this image."

In [6]:
interface = gr.Interface(
    fn=extract_urdu_text,
    inputs=gr.Image(
        type="pil",
        label="Upload Urdu Image"
    ),
    outputs=gr.Textbox(
        label="Extracted Urdu Text",
        lines=5
    ),
    title="Urdu OCR - Code Saviours SI-26",
    description="Upload an image containing Urdu text and get the extracted text."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6ff75ed3ad3491a204.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
